In [16]:
import numpy as np
import pandas as pd
import re

from sklearn.metrics import (
    recall_score,
    precision_score,
    f1_score,
    balanced_accuracy_score,
    accuracy_score,
)

from analysis_shared_functions import (
    parse_heatmap_matrix_crosscheckfingerprints,
    long_df_to_matrix_crosscheckfingerprints,
    parse_sample_matching_results_crosscheckfingerprints,
    calculate_accuracy_metrics_pseudobulk
)
from visualization import all_samples_matrix_viz, bulk_vs_singlecell_matrix_viz


In [2]:
DATA_PATH = "../../data/output_data/"
FIGURES_PATH = "../../data/figures/"
tool = "CrosscheckFingerprints"
dataset = "low_grade_glioma"
pseudobulk = False
ncells = "null"
rd = 1

# Data munging

In [47]:
df, matrix = parse_heatmap_matrix_crosscheckfingerprints(DATA_PATH, pseudobulk, dataset, ncells, rd)

In [48]:
df.head()

,LEFT_SAMPLE,RIGHT_SAMPLE,LOD_SCORE,RESULT
0,GSM7326884_single-cell,GSM7326884_single-cell,3.255386,EXPECTED_MATCH
1,GSM7326884_single-cell,GSM7326913_bulk,-0.507481,EXPECTED_MISMATCH
2,GSM7326884_single-cell,GSM7326893_single-cell,0.688052,UNEXPECTED_MATCH
3,GSM7326884_single-cell,GSM7326885_single-cell,0.427261,UNEXPECTED_MATCH
4,GSM7326884_single-cell,GSM7326883_single-cell,-0.381197,EXPECTED_MISMATCH


In [49]:
matrix = matrix[matrix.index.str.contains("bulk")]
matrix = matrix.loc[:, matrix.columns.str.contains("single-cell")]

In [50]:
matrix

,GSM7326880_single-cell,GSM7326881_single-cell,GSM7326883_single-cell,GSM7326884_single-cell,GSM7326885_single-cell,GSM7326886_single-cell,GSM7326887_single-cell,GSM7326888_single-cell,GSM7326890_single-cell,GSM7326891_single-cell,...,GSM7326894_single-cell,GSM7326895_single-cell,GSM7326896_single-cell,GSM7326898_single-cell,GSM7326899_single-cell,GSM7326900_single-cell,GSM7326901_single-cell,GSM7326902_single-cell,GSM7326903_single-cell,GSM7326905_single-cell
LEFT_SAMPLE,,,,,,,,,,,,,,,,,,,,,
GSM7326906_bulk,0.000000,0.000000,1.599053,0.000000,0.325244,0.301517,-2.064893,-0.774054,0.817915,0.000000,...,0.598772,0.988451,-0.329292,-0.886813,1.494955,1.018979,-1.135701,2.074723,0.268461,-0.012391
GSM7326907_bulk,-2.330802,0.245428,1.118751,0.656273,1.289765,0.705372,-0.295783,-1.218727,-2.233675,0.000000,...,-0.483960,0.013125,-1.980318,-0.428383,3.351797,1.002628,0.079742,-0.219845,0.736982,1.522613
GSM7326908_bulk,0.000000,0.000000,-2.125932,0.000000,-2.976259,0.000000,1.221735,-2.099077,0.420599,0.000000,...,-3.291369,-2.099077,-1.945722,0.088702,-1.712906,-1.963543,-0.900559,-2.247349,-1.945722,-3.558700
GSM7326909_bulk,0.329098,0.254490,-2.352596,-1.790747,0.107853,-4.520066,1.481183,2.674117,-1.715714,0.000000,...,0.139716,2.657232,5.506243,4.179543,2.499354,-1.732030,2.911506,-0.007966,-1.577501,3.121588
GSM7326910_bulk,0.329098,0.000000,0.237962,-2.050097,1.588631,-2.330802,0.863352,1.743968,-3.870687,0.000000,...,1.975658,3.269702,2.401440,5.027975,1.423352,-0.371948,1.889600,1.225300,-1.302649,2.943891
GSM7326912_bulk,-2.330802,0.250071,0.150944,0.661031,-0.651246,0.403856,4.630692,1.222156,3.362099,0.236166,...,-0.305246,-2.796039,-2.889671,-5.522439,0.051438,2.545403,-4.145349,0.960830,-0.089684,0.827976
GSM7326913_bulk,-2.330802,0.000000,-1.635593,-0.507481,-1.232281,0.403856,0.433220,-0.631014,-3.545891,-1.948523,...,-0.183768,-3.072140,-3.974590,-7.218149,1.707022,1.104248,-2.286143,-0.775802,1.787434,0.563219
GSM7326914_bulk,0.000000,0.000000,1.702377,0.000000,1.099910,0.000000,1.978793,2.739336,0.744013,0.232149,...,2.352200,2.634937,3.608134,0.416323,1.529325,0.032593,-0.217860,3.839705,0.425845,2.216564


In [51]:
inferred_matches = parse_sample_matching_results_crosscheckfingerprints(DATA_PATH, pseudobulk, dataset, ncells, rd)
inferred_matches = inferred_matches[inferred_matches.index.str.contains("bulk")]
inferred_matches = inferred_matches.loc[:, inferred_matches.columns.str.contains("single-cell")]
inferred_matches.head()

,GSM7326880_single-cell,GSM7326881_single-cell,GSM7326883_single-cell,GSM7326884_single-cell,GSM7326885_single-cell,GSM7326886_single-cell,GSM7326887_single-cell,GSM7326888_single-cell,GSM7326890_single-cell,GSM7326891_single-cell,...,GSM7326894_single-cell,GSM7326895_single-cell,GSM7326896_single-cell,GSM7326898_single-cell,GSM7326899_single-cell,GSM7326900_single-cell,GSM7326901_single-cell,GSM7326902_single-cell,GSM7326903_single-cell,GSM7326905_single-cell
LEFT_SAMPLE,,,,,,,,,,,,,,,,,,,,,
GSM7326906_bulk,NaN,NaN,1,NaN,1,1,0,0,1,NaN,...,1,1,0,0,1,1,0,1,1,0
GSM7326907_bulk,0,1,1,1,1,1,0,0,0,NaN,...,0,1,0,0,1,1,1,0,1,1
GSM7326908_bulk,NaN,NaN,0,NaN,0,NaN,1,0,1,NaN,...,0,0,0,1,0,0,0,0,0,0
GSM7326909_bulk,1,1,0,0,1,0,1,1,0,NaN,...,1,1,1,1,1,0,1,0,0,1
GSM7326910_bulk,1,NaN,1,0,1,0,1,1,0,NaN,...,1,1,1,1,1,0,1,1,0,1


In [52]:
inferred_matches.shape

(8, 22)

# Real data viz & analysis

In [36]:
bulk_vs_singlecell_matrix_viz(
    matrix,
    pseudobulk=pseudobulk,
    tool=tool,
    dataset=dataset,
    ncells=ncells,
    rd=rd,
    save_fig=True,
    FIGURES_PATH=FIGURES_PATH
)

## Comparison with expected matches

In [53]:
expected_matches_df = pd.read_csv(
    DATA_PATH + f"../metadata/sample_matches/sample_matches_{dataset}.csv"
).dropna(how="all", axis=1).dropna(how="any")
# add _bulk to the bulk sample ids and _single-cell to the single cell sample ids
expected_matches_df["bulk"] = expected_matches_df["bulk"].apply(lambda x: x + "_bulk")
expected_matches_df["single-cell"] = expected_matches_df["single-cell"].apply(lambda x: x + "_single-cell")

In [57]:
expected_matches_df.head()

,bulk,single-cell
2,GSM7326906_bulk,GSM7326883_single-cell
5,GSM7326907_bulk,GSM7326886_single-cell
9,GSM7326908_bulk,GSM7326891_single-cell
14,GSM7326909_bulk,GSM7326896_single-cell
15,GSM7326910_bulk,GSM7326898_single-cell


In [56]:
inferred_matches

,GSM7326880_single-cell,GSM7326881_single-cell,GSM7326883_single-cell,GSM7326884_single-cell,GSM7326885_single-cell,GSM7326886_single-cell,GSM7326887_single-cell,GSM7326888_single-cell,GSM7326890_single-cell,GSM7326891_single-cell,...,GSM7326894_single-cell,GSM7326895_single-cell,GSM7326896_single-cell,GSM7326898_single-cell,GSM7326899_single-cell,GSM7326900_single-cell,GSM7326901_single-cell,GSM7326902_single-cell,GSM7326903_single-cell,GSM7326905_single-cell
LEFT_SAMPLE,,,,,,,,,,,,,,,,,,,,,
GSM7326906_bulk,NaN,NaN,1,NaN,1,1,0,0,1,NaN,...,1,1,0,0,1,1,0,1,1,0
GSM7326907_bulk,0,1,1,1,1,1,0,0,0,NaN,...,0,1,0,0,1,1,1,0,1,1
GSM7326908_bulk,NaN,NaN,0,NaN,0,NaN,1,0,1,NaN,...,0,0,0,1,0,0,0,0,0,0
GSM7326909_bulk,1,1,0,0,1,0,1,1,0,NaN,...,1,1,1,1,1,0,1,0,0,1
GSM7326910_bulk,1,NaN,1,0,1,0,1,1,0,NaN,...,1,1,1,1,1,0,1,1,0,1
GSM7326912_bulk,0,1,1,1,0,1,1,1,1,1,...,0,0,0,0,1,1,0,1,0,1
GSM7326913_bulk,0,NaN,0,0,0,1,1,0,0,0,...,0,0,0,0,1,1,0,0,1,1
GSM7326914_bulk,NaN,NaN,1,NaN,1,NaN,1,1,1,1,...,1,1,1,1,1,1,0,1,1,1


In [58]:
def expected_matches_to_matrix(inferred_matches, expected_matches_df):
    # Create a matrix with the same shape as inferred_matches, filled with zeros
    expected_matrix = pd.DataFrame(
        np.zeros(inferred_matches.shape, dtype=int),
        index=inferred_matches.index,
        columns=inferred_matches.columns
    )
    # Iterate through the expected matches and set the corresponding entries in the matrix to 1
    for _, row in expected_matches_df.iterrows():
        bulk_sample = row['bulk']
        single_cell_sample = row['single-cell']
        if bulk_sample in expected_matrix.index and single_cell_sample in expected_matrix.columns:
            expected_matrix.loc[bulk_sample, single_cell_sample] = 1

    return expected_matrix

In [60]:
expected_matches = expected_matches_to_matrix(inferred_matches,expected_matches_df)

In [64]:
expected_matches

,GSM7326880_single-cell,GSM7326881_single-cell,GSM7326883_single-cell,GSM7326884_single-cell,GSM7326885_single-cell,GSM7326886_single-cell,GSM7326887_single-cell,GSM7326888_single-cell,GSM7326890_single-cell,GSM7326891_single-cell,...,GSM7326894_single-cell,GSM7326895_single-cell,GSM7326896_single-cell,GSM7326898_single-cell,GSM7326899_single-cell,GSM7326900_single-cell,GSM7326901_single-cell,GSM7326902_single-cell,GSM7326903_single-cell,GSM7326905_single-cell
LEFT_SAMPLE,,,,,,,,,,,,,,,,,,,,,
GSM7326906_bulk,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GSM7326907_bulk,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GSM7326908_bulk,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
GSM7326909_bulk,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
GSM7326910_bulk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
GSM7326912_bulk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
GSM7326913_bulk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
GSM7326914_bulk,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [73]:
def calculate_accuracy_real_data(expected_matches, inferred_matches):

    accuracy = accuracy_score(inferred_matches.fillna(0).astype(int).values.flatten(), expected_matches.values.flatten())
    precision = precision_score(inferred_matches.fillna(0).astype(int).values.flatten(), expected_matches.values.flatten())
    recall = recall_score(inferred_matches.fillna(0).astype(int).values.flatten(), expected_matches.values.flatten())
    f1 = f1_score(inferred_matches.fillna(0).astype(int).values.flatten(), expected_matches.values.flatten())
    balanced_accuracy = balanced_accuracy_score(inferred_matches.fillna(0).astype(int).values.flatten(), expected_matches.values.flatten())

    return accuracy, balanced_accuracy, precision, recall, f1

In [74]:
calculate_accuracy_real_data(expected_matches, inferred_matches)

(0.5511363636363636,
 0.5356819650937298,
 0.875,
 0.08235294117647059,
 0.15053763440860216)

In [81]:
# Which sample matches from the expected matches were correctly identified by the tool, 
# and which were missed? And which incorrect matches were made by the tool?
# I.e. which entries in the expected_matches matrix are 1 and also 1 in the inferred_matches matrix (true positives),
# which entries in the expected_matches matrix are 1 but 0 in the inferred_matches matrix (false negatives),
# and which entries in the expected_matches matrix are 0 but 1 in the inferred_matches matrix (false positives)
tp = ((inferred_matches.fillna(0).astype(int) == 1) & (expected_matches == 1)).sum().sum()
fn = ((inferred_matches.fillna(0).astype(int) == 0) & (expected_matches == 1)).sum().sum()
fp = ((inferred_matches.fillna(0).astype(int) == 1) & (expected_matches == 0)).sum().sum()
tn = ((inferred_matches.fillna(0).astype(int) == 0) & (expected_matches == 0)).sum().sum()

In [83]:
tp, fn, fp, tn

(np.int64(7), np.int64(1), np.int64(78), np.int64(90))